In [21]:
import duckdb as db
from datetime import datetime
from dateutil.relativedelta import relativedelta

In [2]:
tpa = db.read_csv('../../data/TrafficPerAirport.csv')
airports = db.read_csv('../../data/Airport.csv')
airservice = db.read_csv('../../data/AirService.csv')
aircraftmov = db.read_csv('../../data/AircraftMovement.csv')

In [3]:
db.sql("SELECT AirService, AircraftMovement, Month, a.AirportName AS BaseAirport, a.Latitude AS BaseLatitude, a.Longitude AS BaseLongitude, \
        a2.AirportName AS StopoverAirport, a2.Latitude AS StopoverLatitude, a2.Longitude AS StopoverLongitude, \
        Passengers, Operations, Goods, Mail  \
        FROM tpa t INNER JOIN airports a ON a.AirportId = t.BaseAirportId  \
        INNER JOIN airports a2 ON a2.AirportId = t.StopoverAirportId \
        INNER JOIN airservice USING(AirServiceId)\
        INNER JOIN aircraftmov USING (AircraftMovementId)")

┌───────────────┬──────────────────┬────────────┬───────────────────────┬──────────────┬───────────────┬──────────────────────────────┬──────────────────┬───────────────────┬────────────┬────────────┬───────┬───────┐
│  AirService   │ AircraftMovement │   Month    │      BaseAirport      │ BaseLatitude │ BaseLongitude │       StopoverAirport        │ StopoverLatitude │ StopoverLongitude │ Passengers │ Operations │ Goods │ Mail  │
│    varchar    │     varchar      │    date    │        varchar        │    double    │    double     │           varchar            │      double      │      double       │   int64    │   int64    │ int64 │ int64 │
├───────────────┼──────────────────┼────────────┼───────────────────────┼──────────────┼───────────────┼──────────────────────────────┼──────────────────┼───────────────────┼────────────┼────────────┼───────┼───────┤
│ Commercial    │ Arrival          │ 2004-01-01 │ Fuerteventura Airport │      28.4527 │      -13.8638 │ Graz Airport               

In [4]:
import pandas as pd
import json

def create_air_traffic_html(df, output_file="air_traffic.html", div_id="air_traffic_div"):
    # Ensure Month is datetime
    df = df.copy()
    df["Month"] = pd.to_datetime(df["Month"])

    # Prepare minimal records list for embedding in HTML/JS
    records = []
    for _, r in df.iterrows():
        records.append({
            "AirService": None if pd.isna(r["AirService"]) else str(r["AirService"]),
            "AircraftMovement": None if pd.isna(r["AircraftMovement"]) else str(r["AircraftMovement"]),
            "Month": r["Month"].strftime("%Y-%m-%d"),
            "BaseAirport": None if pd.isna(r["BaseAirport"]) else str(r["BaseAirport"]),
            "BaseLatitude": None if pd.isna(r["BaseLatitude"]) else float(r["BaseLatitude"]),
            "BaseLongitude": None if pd.isna(r["BaseLongitude"]) else float(r["BaseLongitude"]),
            "StopoverAirport": None if pd.isna(r["StopoverAirport"]) else str(r["StopoverAirport"]),
            "StopoverLatitude": None if pd.isna(r["StopoverLatitude"]) else float(r["StopoverLatitude"]),
            "StopoverLongitude": None if pd.isna(r["StopoverLongitude"]) else float(r["StopoverLongitude"]),
            "Passengers": 0 if pd.isna(r.get("Passengers", 0)) else int(r.get("Passengers", 0)),
            "Operations": 0 if pd.isna(r.get("Operations", 0)) else int(r.get("Operations", 0)),
            "Goods": 0 if pd.isna(r.get("Goods", 0)) else int(r.get("Goods", 0)),
            "Mail": 0 if pd.isna(r.get("Mail", 0)) else int(r.get("Mail", 0))
        })

    raw_json = json.dumps(records)

    min_month = df["Month"].min().strftime("%Y-%m")
    max_month = df["Month"].max().strftime("%Y-%m")

    # f-string template with corrected end-of-month computation in JS
    html = f"""<!doctype html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Air traffic geo plot</title>
  <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 12px; }}
    .controls {{ display:flex; flex-wrap:wrap; gap:10px; align-items:center; margin-bottom:8px; }}
    .control {{ display:flex; flex-direction:column; font-size:13px; }}
    label {{ font-size:12px; margin-bottom:4px; color:#333; }}
    select, input[type="month"], button {{ padding:6px 8px; font-size:13px; }}
    #{div_id} {{ width:100%; height:640px; }}
    .empty-note {{ color:#666; margin-top:8px; }}
  </style>
</head>
<body>
  <h3 style="text-align:center;margin-top:0;">Passengers Map</h3>
  <div class="controls">
    <div class="control">
      <label>Metric</label>
      <select id="metricSel">
        <option value="Passengers">Passengers</option>
        <option value="Operations">Operations</option>
        <option value="Goods">Goods</option>
        <option value="Mail">Mail</option>
      </select>
    </div>

    <div class="control">
      <label>AirService</label>
      <select id="airServiceSel"><option>All</option></select>
    </div>

    <div class="control">
      <label>AircraftMovement</label>
      <select id="aircraftMovementSel"><option>All</option></select>
    </div>

    <div class="control">
      <label>BaseAirport</label>
      <select id="baseAirportSel"><option>All</option></select>
    </div>

    <div class="control">
      <label>StopoverAirport</label>
      <select id="stopoverAirportSel"><option>All</option></select>
    </div>

    <div class="control">
      <label>Start month</label>
      <input id="startMonth" type="month" min="{min_month}" max="{max_month}" value="{min_month}">
    </div>

    <div class="control">
      <label>End month</label>
      <input id="endMonth" type="month" min="{min_month}" max="{max_month}" value="{max_month}">
    </div>

    <div class="control" style="margin-left:8px;">
      <label>&nbsp;</label>
      <button id="resetBtn">Reset</button>
    </div>
  </div>

  <div id="{div_id}"></div>
  <div class="empty-note" id="emptyNote" style="display:none">No data for the selected filters / date range.</div>

<script>
const rawData = {raw_json};   // embedded records from Python
const plotDiv = document.getElementById("{div_id}");

function uniqSorted(arr) {{
  return Array.from(new Set(arr.filter(x => x !== null && x !== undefined))).sort();
}}

function populateSelect(id, values) {{
  const sel = document.getElementById(id);
  while (sel.options.length > 1) sel.remove(1);
  values.forEach(v => {{
    const opt = document.createElement("option");
    opt.value = v;
    opt.text = v;
    sel.appendChild(opt);
  }});
}}

function initControls() {{
  populateSelect("airServiceSel", uniqSorted(rawData.map(d => d.AirService)));
  populateSelect("aircraftMovementSel", uniqSorted(rawData.map(d => d.AircraftMovement)));
  populateSelect("baseAirportSel", uniqSorted(rawData.map(d => d.BaseAirport)));
  populateSelect("stopoverAirportSel", uniqSorted(rawData.map(d => d.StopoverAirport)));

  document.getElementById("metricSel").addEventListener("change", updatePlot);
  document.getElementById("airServiceSel").addEventListener("change", updatePlot);
  document.getElementById("aircraftMovementSel").addEventListener("change", updatePlot);
  document.getElementById("baseAirportSel").addEventListener("change", updatePlot);
  document.getElementById("stopoverAirportSel").addEventListener("change", updatePlot);
  document.getElementById("startMonth").addEventListener("change", updatePlot);
  document.getElementById("endMonth").addEventListener("change", updatePlot);
  document.getElementById("resetBtn").addEventListener("click", resetFilters);
}}

function resetFilters() {{
  document.getElementById("metricSel").value = "Passengers";
  document.getElementById("airServiceSel").value = "All";
  document.getElementById("aircraftMovementSel").value = "All";
  document.getElementById("baseAirportSel").value = "All";
  document.getElementById("stopoverAirportSel").value = "All";
  document.getElementById("startMonth").value = "{min_month}";
  document.getElementById("endMonth").value = "{max_month}";
  updatePlot();
}}

function lineWidthForValue(v) {{
  if (v <= 0) return 1;
  return Math.min(12, Math.max(1, Math.log(v + 1) * 2.0));
}}
function markerSizeForValue(v) {{
  if (v <= 0) return 4;
  return Math.min(18, Math.max(4, Math.log(v + 1) * 3.0));
}}

function updatePlot() {{
  const metric = document.getElementById("metricSel").value;
  const airService = document.getElementById("airServiceSel").value;
  const aircraftMovement = document.getElementById("aircraftMovementSel").value;
  const baseAirport = document.getElementById("baseAirportSel").value;
  const stopoverAirport = document.getElementById("stopoverAirportSel").value;

  // read month inputs; fallback to min/max if missing
  const startInput = document.getElementById("startMonth").value || "{min_month}";
  const endInput = document.getElementById("endMonth").value || "{max_month}";

  // Build startDate at start of month
  const startDate = new Date(startInput + "-01T00:00:00");

  // Build endDate robustly: start of month then advance one month and subtract 1ms
  const endDate = new Date(endInput + "-01T00:00:00");
  endDate.setMonth(endDate.getMonth() + 1);
  endDate.setMilliseconds(endDate.getMilliseconds() - 1);

  // filter rows (inclusive start..end)
  const filtered = rawData.filter(d => {{
    const m = new Date(d.Month);
    if (isNaN(m)) return false;
    if (m < startDate || m > endDate) return false;
    if (airService !== "All" && d.AirService !== airService) return false;
    if (aircraftMovement !== "All" && d.AircraftMovement !== aircraftMovement) return false;
    if (baseAirport !== "All" && d.BaseAirport !== baseAirport) return false;
    if (stopoverAirport !== "All" && d.StopoverAirport !== stopoverAirport) return false;
    return true;
  }});

  // aggregate by base-stop pair
  const groups = {{}};
  filtered.forEach(d => {{
    if (d.BaseLatitude == null || d.BaseLongitude == null || d.StopoverLatitude == null || d.StopoverLongitude == null) return;
    const key = [d.BaseAirport, d.BaseLatitude, d.BaseLongitude, d.StopoverAirport, d.StopoverLatitude, d.StopoverLongitude].join("||");
    if (!groups[key]) {{
      groups[key] = {{
        BaseAirport: d.BaseAirport,
        BaseLatitude: +d.BaseLatitude,
        BaseLongitude: +d.BaseLongitude,
        StopoverAirport: d.StopoverAirport,
        StopoverLatitude: +d.StopoverLatitude,
        StopoverLongitude: +d.StopoverLongitude,
        value: 0
      }};
    }}
    const add = Number(d[metric] || 0);
    groups[key].value += isNaN(add) ? 0 : add;
  }});

  const groupList = Object.values(groups).sort((a,b) => b.value - a.value);

  const traces = [];
  groupList.forEach(g => {{
    const val = g.value;
    const lineW = lineWidthForValue(val);
    const mkSize = markerSizeForValue(val);

    traces.push({{
      type: "scattergeo",
      mode: "lines+markers",
      lon: [g.BaseLongitude, g.StopoverLongitude],
      lat: [g.BaseLatitude, g.StopoverLatitude],
      text: `${{g.BaseAirport}} → ${{g.StopoverAirport}}<br>${{metric}}: ${{val}}`,
      hoverinfo: "text",
      line: {{ width: lineW, color: "rgba(31,119,180,0.9)" }},
      marker: {{ size: mkSize, symbol: "circle", opacity: 0.9 }},
      name: `${{g.BaseAirport}} → ${{g.StopoverAirport}} (${{val}})`
    }});
  }});

  const airportAgg = {{}};
  groupList.forEach(g => {{
    if (!airportAgg[g.BaseAirport]) airportAgg[g.BaseAirport] = {{ lat: g.BaseLatitude, lon: g.BaseLongitude, total: 0 }};
    if (!airportAgg[g.StopoverAirport]) airportAgg[g.StopoverAirport] = {{ lat: g.StopoverLatitude, lon: g.StopoverLongitude, total: 0 }};
    airportAgg[g.BaseAirport].total += g.value;
    airportAgg[g.StopoverAirport].total += g.value;
  }});
  const airports = Object.keys(airportAgg);
  if (airports.length) {{
    traces.push({{
      type: "scattergeo",
      mode: "markers",
      lon: airports.map(a => airportAgg[a].lon),
      lat: airports.map(a => airportAgg[a].lat),
      text: airports.map(a => `${{a}}<br>${{metric}}: ${{airportAgg[a].total}}`),
      hoverinfo: "text",
      marker: {{
        size: airports.map(a => markerSizeForValue(airportAgg[a].total) + 2),
        symbol: "circle",
        line: {{ width: 0.5, color: "#333" }}
      }},
      name: "Airports"
    }});
  }}

  const layout = {{
    title: `Connections — ${{metric}} (sum over range)`,
    geo: {{
      scope: "world",
      projection: {{ type: "natural earth" }},
      showland: true,
      landcolor: "rgb(240,240,240)",
      showcountries: true,
      countrycolor: "rgb(200,200,200)"
    }},
    margin: {{ t: 40, b: 20, l: 0, r: 0 }},
    legend: {{ orientation: "h", y: -0.05 }}
  }};

  if (traces.length === 0) {{
    document.getElementById("emptyNote").style.display = "block";
    Plotly.react(plotDiv, [], layout, {{displayModeBar: true}});
  }} else {{
    document.getElementById("emptyNote").style.display = "none";
    Plotly.react(plotDiv, traces, layout, {{displayModeBar: true}});
  }}
}}

initControls();
resetFilters();
</script>
</body>
</html>
"""

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"Wrote interactive map to {output_file}")

"""
# -------------------------
# Example usage with user's example rows:
if True:
    # sample dataframe (replace with your real df)
    data = {
        "AirService": ["Commercial", "Commercial", "Commercial"],
        "AircraftMovement": ["Arrival", "Arrival", "Arrival"],
        "Month": ["2004-01-01", "2004-02-01", "2004-03-01"],
        "BaseAirport": ["Fuerteventura Airport"]*3,
        "BaseLatitude": [28.4527]*3,
        "BaseLongitude": [-13.8638]*3,
        "StopoverAirport": ["Graz Airport"]*3,
        "StopoverLatitude": [46.9911]*3,
        "StopoverLongitude": [15.4396]*3,
        "Passengers": [10, 5, 3],
        "Operations": [1, 2, 1],
        "Goods": [0, 2, 0],
        "Mail": [0, 0, 5],
    }
    df = pd.DataFrame(data)
    create_air_traffic_html(df, output_file="air_traffic.html", div_id="air_traffic_div")
"""

'\n# -------------------------\n# Example usage with user\'s example rows:\nif True:\n    # sample dataframe (replace with your real df)\n    data = {\n        "AirService": ["Commercial", "Commercial", "Commercial"],\n        "AircraftMovement": ["Arrival", "Arrival", "Arrival"],\n        "Month": ["2004-01-01", "2004-02-01", "2004-03-01"],\n        "BaseAirport": ["Fuerteventura Airport"]*3,\n        "BaseLatitude": [28.4527]*3,\n        "BaseLongitude": [-13.8638]*3,\n        "StopoverAirport": ["Graz Airport"]*3,\n        "StopoverLatitude": [46.9911]*3,\n        "StopoverLongitude": [15.4396]*3,\n        "Passengers": [10, 5, 3],\n        "Operations": [1, 2, 1],\n        "Goods": [0, 2, 0],\n        "Mail": [0, 0, 5],\n    }\n    df = pd.DataFrame(data)\n    create_air_traffic_html(df, output_file="air_traffic.html", div_id="air_traffic_div")\n'

In [20]:
max_date_local = db.sql(' \
SELECT MAX(Month) FROM tpa \
').fetchone()[0]

max_date_local


datetime.date(2025, 7, 1)

In [24]:
one_year_ago = max_date_local - relativedelta(years=1)

one_year_ago

datetime.date(2024, 7, 1)

In [25]:
create_air_traffic_html(db.sql(f"SELECT AirService, AircraftMovement, Month, a.AirportName AS BaseAirport, a.Latitude AS BaseLatitude, a.Longitude AS BaseLongitude, \
        a2.AirportName AS StopoverAirport, a2.Latitude AS StopoverLatitude, a2.Longitude AS StopoverLongitude, \
        Passengers, Operations, Goods, Mail  \
        FROM tpa t INNER JOIN airports a ON a.AirportId = t.BaseAirportId  \
        INNER JOIN airports a2 ON a2.AirportId = t.StopoverAirportId \
        INNER JOIN airservice USING(AirServiceId)\
        INNER JOIN aircraftmov USING (AircraftMovementId)\
        WHERE Month >= {one_year_ago}").df(),
        output_file="air_traffic.html", 
        div_id="air_traffic_div")

BinderException: Binder Error: Cannot compare values of type DATE and type INTEGER - an explicit cast is required